## 📝 Chap06-2. Make Quize With Image
#### 문제 생성 함수 만들기

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")  # 환경 변수에서 API 키 가져오기
client = OpenAI(api_key=api_key)  # OpenAI 클라이언트의 인스턴스 생성

def make_quiz_from_image(image_path):

    quiz_prompt = """
    제공된 이미지를 바탕으로, 다음과 같은 양식으로 퀴즈를 만들어주세요. 
    정답은 1~4 중 하나만 해당하도록 출제하세요.
    아래는 예시입니다. 
    ----- 예시 -----

    Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
    - (1) 왼쪽에서 첫번째 여성은 흰색 티셔츠를 입고 있습니다.
    - (2) 왼쪽에서 두번째 여성은 노란색 음료를 들고 있습니다.
    - (3) 왼쪽에서 세번째 여성은 선글라스를 착용하고 있습니다.
    - (4) 왼쪽에서 네번째 여성은 검은색 음료를 들고 있습니다.
        
    정답: (4) 왼쪽에서 네번째 여성은 검은색이 아니라 주황색 음료를 들고 있습니다.
    (주의: 정답은 1~4 중 하나만 선택되도록 출제하세요.)
    ======
    """

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": quiz_prompt},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_path,
                    },
                },
            ],
        }
    ]

    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=messages
    )

    return response.choices[0].message.content

q = make_quiz_from_image("https://images.unsplash.com/photo-1532635241-17e820acc59f?q=80&w=1115&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D")

In [3]:
print(q)

Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?

- (1) 왼쪽에서 첫 번째 여성은 흰색 티셔츠를 입고 있습니다.
- (2) 왼쪽에서 두 번째 여성은 검은색 민소매 상의를 입고 있습니다.
- (3) 왼쪽에서 세 번째 여성은 연한 파란색 셔츠를 입고 있습니다.
- (4) 오른쪽의 여성은 파란색 음료가 담긴 잔을 들고 있습니다.

정답: (4) 오른쪽의 여성은 파란색 음료가 아니라 붉은 주황색 음료가 담긴 잔을 들고 있습니다.


In [4]:
q = make_quiz_from_image("https://images.unsplash.com/photo-1522202176988-66273c2fd55f?q=80&w=1171&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D")
print(q)

Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?

- (1) 왼쪽의 여성은 노트북을 사용하고 있습니다.
- (2) 가운데의 여성은 웃고 있습니다.
- (3) 오른쪽의 남성은 안경을 쓰고 있습니다.
- (4) 오른쪽의 남성은 빨간색 셔츠를 입고 있지 않습니다.

정답: (4) 오른쪽의 남성은 빨간색 셔츠를 입고 있습니다.


---
#### 여러 이미지로 문제집 만들기

In [5]:
# 웹 이미지 URL 5개
urls = [
    "https://images.unsplash.com/photo-1511988617509-a57c8a288659?q=80&w=1171&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://images.unsplash.com/photo-1504022462188-88f023db97bf?q=80&w=1170&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://images.unsplash.com/photo-1517486808906-6ca8b3f04846?q=80&w=749&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://images.unsplash.com/photo-1549057446-9f5c6ac91a04?q=80&w=1634&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://images.unsplash.com/photo-1532675432006-329c6fed7045?q=80&w=627&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
]

In [8]:
txt = '' # ①  문제들을 계속 붙여 나가기 위해 빈 문자열 선언
no = 1 # 문제 번호를 위해 선언
for g in urls:
    try:
        q = make_quiz_from_image(g)  # 문제 출제 (외부 API 호출이므로 예외 처리)
    except Exception as e:
        print(e)
        continue

    divider = f'## 문제 {no}\n\n'
    print(divider)

    txt += divider  # ③

    # URL에서 파일명 추출(마크다운 레이블 용)
    label = f'문제 {no}'
    txt += f'![{label}]({g})\n\n'  # ③ 웹 이미지를 그대로 링크로 삽입

    # 문제 추가
    print(q)
    txt += q + '\n\n---------------------\n\n'

    # ④ 마크다운 파일로 저장 (루프마다 덮어쓰기 -> 현재까지 누적된 내용 저장)
    with open('data/images/image_quiz.md', 'w', encoding='utf-8') as f:
        f.write(txt)

    no += 1  # 문제 번호 증가

## 문제 1


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?

- (1) 왼쪽의 남성은 안경을 쓰고 흰색 셔츠를 입고 있습니다.
- (2) 왼쪽에서 두 번째 여성은 선글라스를 쓰고 청색 데님 재킷을 입고 있습니다.
- (3) 왼쪽에서 세 번째 남성은 선글라스를 쓰고 목에 스카프를 두르고 있습니다.
- (4) 오른쪽의 남성은 긴소매 셔츠를 입고 양손을 내리고 있습니다.

정답: (4) 오른쪽의 남성은 긴소매 셔츠가 아니라 민소매 상의를 입고 있으며, 한 손을 들어 올리고 있습니다.
## 문제 2


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?

- (1) 왼쪽 남성은 안경을 쓰고 있습니다.
- (2) 가운데 여성은 주황색 상의를 입고 있습니다.
- (3) 오른쪽 여성은 검은색 재킷을 입고 있습니다.
- (4) 오른쪽 여성은 파란색 재킷을 입고 있습니다.

정답: (4) 오른쪽 여성은 파란색이 아니라 검은색 재킷을 입고 있습니다.
## 문제 3


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?

- (1) 왼쪽에서 첫 번째 사람은 빨간색 상의를 입고 있습니다.
- (2) 왼쪽에서 두 번째 사람은 파란색 셔츠를 입고 있습니다.
- (3) 왼쪽에서 세 번째 사람은 흰색 상의를 입고 있습니다.
- (4) 오른쪽에서 첫 번째 사람은 흰색 셔츠를 입고 있습니다.

정답: (4) 오른쪽에서 첫 번째 사람은 흰색이 아니라 짙은 남색 상의를 입고 있습니다.
## 문제 4


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?

- (1) 왼쪽에서 첫 번째 남성은 갈색 재킷과 분홍색 티셔츠를 입고 있습니다.
- (2) 왼쪽에서 두 번째 남성은 청색 데님 재킷을 입고 있습니다.
- (3) 왼쪽에서 세 번째 여성은 초록색과 보라색이 섞인 스웨터를 입고 있습니다.
- (4) 왼쪽에서 네 번째 여성은 빨간색 스웨터를 입고 있습니다.

정답: (4) 왼쪽에서 네 번째 여성은 빨간색이 아니라 노란색 스웨터를 입고 있습니다.
#